In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 06
# Architectural Awareness: Building the Sensor into the Boat
# =============================================================
#
# Series:  Humble Model / Architectural Awareness
# Dataset: CIFAR-10
# Model:   CNN (same as Essay #5)
#
# Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10)
#   Part C: Model Architecture (shared backbone)
#   Part D: Train or load all three models
#   Part E: Attacks
#   Part E: Attacks--Report
#   Part F: Signal collection
#   Part G: Calibrate thresholds ON CALIB DATA ONLY
#   Part H: Head-to-Head Comparison
#   Part H: Head-to-Head Comparison--Report 
#   Part I: Fused Gate Evaluation 
#   Part J: Final Results and Summary
# =============================================================

In [ ]:
# =============================================================
# Adversarial Attacks Series -- Note 06 (REVISED)
# Architectural Awareness: Building the Sensor into the Boat
# =============================================================
#
# Series:  Humble Model / Architectural Awareness
# Dataset: CIFAR-10
# Model:   CNN (same backbone as Essay #5)
#
# CHANGES IN THIS REVISION (see chat for the full explanation of each):
#   1. RISK FORMULA FIXED. The previous version computed:
#        risk = 100 * ((total - correct) + deferred) / (total + deferred)
#      which counts every deferred sample as a failure -- a PERFECT
#      gate that defers exactly its own errors would still score
#      risk == deferral_rate under that formula, not ~0%. That
#      inverts this series' entire premise that deferral is the SAFE
#      choice. Risk is now `100 - accuracy(on predicted)`, matching
#      Essays #4 and #5 exactly, so numbers are comparable across the
#      series. Deferral rate and coverage are reported alongside it
#      as separate numbers, the way #4/#5 did -- not folded into risk.
#   2. clamp_valid() used in EVERY attack function. The previous
#      version used torch.clamp(x, 0, 1) in most of them, which is
#      wrong for CIFAR-normalized inputs -- exactly the bug Essay #5
#      fixed, that didn't carry over to this notebook.
#   3. The calibration split is now actually used. Every threshold
#      (confidence, disagreement, evidence) is calculated on
#      calib_loader and applied to eval_loader -- not calculated
#      in-sample on the same data being evaluated.
#   4. Multi-head's threshold is now calibrated (75th percentile of
#      disagreement on the calibration set), not a hardcoded 0.05
#      that never fired.
#   5. Boundary distance is computed for every image actually used in
#      the fused-gate sample -- no more padding unsampled images in a
#      batch with a copy of an unrelated image's score. Sample size
#      is controlled by evaluating fewer total images, not by
#      silently faking values within a batch.
#   6. Part H now includes a STANDALONE boundary-distance row, so its
#      individual contribution can be compared to confidence,
#      disagreement, and evidence before looking at the fused blend.
#   7. Fused gate uses Essay #5's OR-based fusion (defer if ANY
#      individual signal crosses ITS OWN calibrated threshold) instead
#      of a weighted linear blend of differently-scaled raw values.
#      The previous weighted-sum design mixed confidence (0-1, higher
#      = safer), disagreement (unbounded, higher = riskier), evidence
#      (unbounded, higher = safer), and boundary distance (0-1ish,
#      LOWER = riskier) into one number without a principled way to
#      make their scales comparable. OR-based fusion sidesteps that:
#      each signal only needs to be individually well-calibrated.
# =============================================================


In [1]:
# ─────────────────────────────────────────────────────────────
# Part A: Imports and Setup
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

Using device: cpu


In [2]:
# ─────────────────────────────────────────────────────────────
# Part B: Dataset Loading (CIFAR-10) — with held-out split
# ─────────────────────────────────────────────────────────────

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

# Held-out split: 20% calibration, 80% evaluation
calib_size = int(0.2 * len(test_dataset))
eval_size = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(test_dataset, [calib_size, eval_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}")
print(f"Evaluation samples: {len(eval_dataset):,}")

# Per-channel valid normalized range
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)

Training samples: 50,000
Calibration samples: 2,000
Evaluation samples: 8,000


In [3]:
# ─────────────────────────────────────────────────────────────
# Part C: Model Architecture (Shared Backbone)
# ─────────────────────────────────────────────────────────────

class BackboneCNN(nn.Module):
    """Shared backbone for all directions."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return x

class StandardCNN(nn.Module):
    """Standard CNN for baseline (same as Essay #5)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = BackboneCNN()
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.fc_out(features)

In [ ]:
# -------------------------------------------------------------
# Part D: Train or load all three models -- training loop unchanged,
# only the checkpoint filenames matter. If you already have
# checkpoints from the previous run, they're still valid (training
# wasn't buggy, only evaluation/attacks were) -- just point at them.
# -------------------------------------------------------------

def train_standard(model, loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
    return model

def train_multihead(model, loader, epochs=20, num_heads=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(num_heads)]).mean()
            loss.backward()
            optimizer.step()
    return model

def train_evidential(model, loader, epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            alpha = model(images)
            loss = evidential_loss(alpha, F.one_hot(labels, num_classes=10).float())
            loss.backward()
            optimizer.step()
    return model

def load_or_train(model, path, train_fn, loader):
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded {path}")
    else:
        print(f"Training -> {path}")
        model = train_fn(model, loader)
        torch.save({'model_state_dict': model.state_dict()}, path)
    return model

baseline_model = load_or_train(StandardCNN().to(DEVICE), 'checkpoint_baseline_cifar10.pth', train_standard, train_loader)
multi_head_model = load_or_train(MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE), 'checkpoint_multihead.pth', train_multihead, train_loader)
evidential_model = load_or_train(EvidentialCNN(BackboneCNN()).to(DEVICE), 'checkpoint_evidential.pth', train_evidential, train_loader)

for m in [baseline_model, multi_head_model, evidential_model]:
    m.eval()

In [ ]:
# -------------------------------------------------------------
# Part E: Attacks -- ONE set of correct attack functions, reused by
# everything below. clamp_valid() everywhere, no exceptions.
# -------------------------------------------------------------

def fgsm_standard(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(images), labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_standard(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(images_adv), labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def _multihead_loss(model, images, labels):
    logits = model(images)
    criterion = nn.CrossEntropyLoss()
    return torch.stack([criterion(logits[:, h, :], labels) for h in range(logits.size(1))]).mean()

def fgsm_multihead(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    loss = _multihead_loss(model, images, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_multihead(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        loss = _multihead_loss(model, images_adv, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def fgsm_evidential(model, images, labels, epsilon=0.03):
    model.eval()
    images = images.clone().detach().requires_grad_(True)
    alpha = model(images)
    loss = nn.CrossEntropyLoss()(alpha, labels)
    model.zero_grad(); loss.backward()
    return clamp_valid(images + epsilon * images.grad.sign()).detach()

def pgd_evidential(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    model.eval()
    images_adv = images.clone().detach()
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        alpha = model(images_adv)
        loss = nn.CrossEntropyLoss()(alpha, labels)
        model.zero_grad(); loss.backward()
        with torch.no_grad():
            images_adv = images_adv + step_size * images_adv.grad.sign()
            perturbation = torch.clamp(images_adv - images, -epsilon, epsilon)
            images_adv = clamp_valid(images + perturbation)
    return images_adv.detach()

def estimate_boundary_distance(model, image, label, max_iters=30):
    model.eval()
    image = image.clone().detach().requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(image), torch.tensor([label], device=DEVICE))
    model.zero_grad(); loss.backward()
    grad = image.grad.data.sign()
    low, high = 0.0, 1.0
    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = clamp_valid(image + mid * grad)
        with torch.no_grad():
            pred = model(perturbed).argmax().item()
        if pred != label:
            high = mid
        else:
            low = mid
    return (low + high) / 2

PGD_PARAMS = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}


✅ Loaded multi-head model from checkpoint


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part E: Attacks--Report 
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("🔍 Multi-Head CNN Evaluation")
print("="*55)

# ─── Ensure model is loaded ───

if 'multi_head_model' not in locals():
    print("🔄 Loading multi-head model...")
    multi_head_model = MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE)
    
    if os.path.exists('checkpoint_multihead.pth'):
        checkpoint = torch.load('checkpoint_multihead.pth', map_location=DEVICE)
        multi_head_model.load_state_dict(checkpoint['model_state_dict'])
        multi_head_model.eval()  # Important: Set to eval mode
        print("✅ Loaded multi-head model from checkpoint")
    else:
        print("❌ No checkpoint found. Please train the model first.")
        raise SystemExit("Multi-head model not available")

# ─── Attack functions with proper gradient handling ───

def fgsm_attack_multihead(model, images, labels, epsilon=0.03):
    """FGSM attack for multi-head models with proper gradient handling."""
    # Ensure model is in eval mode
    model.eval()
    
    # Clone and detach images, then require gradients
    images_adv = images.clone().detach().requires_grad_(True)
    
    # Forward pass
    logits = model(images_adv)
    criterion = nn.CrossEntropyLoss()
    
    # Average loss across heads
    loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(logits.size(1))]).mean()
    
    # Zero gradients and backward
    model.zero_grad()
    loss.backward()
    
    # Create perturbation
    with torch.no_grad():
        # Get gradient sign
        grad_sign = images_adv.grad.sign()
        # Apply perturbation
        images_adv = images_adv + epsilon * grad_sign
        images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def pgd_attack_multihead(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    """PGD attack for multi-head models with proper gradient handling."""
    # Ensure model is in eval mode
    model.eval()
    
    # Start from clean images
    images_adv = images.clone().detach()
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(num_steps):
        # Require gradients for this iteration
        images_adv.requires_grad_(True)
        
        # Forward pass
        logits = model(images_adv)
        
        # Average loss across heads
        loss = torch.stack([criterion(logits[:, h, :], labels) for h in range(logits.size(1))]).mean()
        
        # Zero gradients and backward
        model.zero_grad()
        loss.backward()
        
        # Update perturbation
        with torch.no_grad():
            # Get gradient sign
            grad_sign = images_adv.grad.sign()
            
            # Take a step
            images_adv = images_adv + step_size * grad_sign
            
            # Project back to epsilon ball
            perturbation = images_adv - images
            perturbation = torch.clamp(perturbation, -epsilon, epsilon)
            images_adv = images + perturbation
            images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def evaluate_multi_head_accuracy(model, loader, attack_type=None, attack_params=None):
    """Evaluate multi-head model accuracy using majority vote."""
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_multihead(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_multihead(model, images, labels, **attack_params)
        
        # Get predictions from all heads
        with torch.no_grad():
            logits = model(images)  # [batch, num_heads, num_classes]
            
            # Majority voting across heads
            preds_per_head = logits.argmax(dim=2)  # [batch, num_heads]
            majority_preds, _ = torch.mode(preds_per_head, dim=1)  # [batch]
            
            correct += (majority_preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(majority_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return 100 * correct / total, all_preds, all_labels

def evaluate_with_deferral(model, loader, threshold, attack_type=None, attack_params=None):
    """Evaluate multi-head model with deferral based on disagreement."""
    model.eval()
    correct = 0
    total = 0
    deferred = 0
    all_preds = []
    all_labels = []
    deferred_indices = []
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        batch_size = images.size(0)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_multihead(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_multihead(model, images, labels, **attack_params)
        
        # Get mean predictions and variance
        with torch.no_grad():
            mean_probs, var_probs = model.forward_with_disagreement(images)
            
            # Compute disagreement (max variance across classes)
            disagreement = var_probs.max(dim=1)[0]  # [batch]
            
            # Get predictions
            preds = mean_probs.argmax(dim=1)  # [batch]
            
            # Defer decisions based on disagreement threshold
            defer_mask = disagreement > threshold
            
            # For non-deferred samples, check accuracy
            non_defer_mask = ~defer_mask
            if non_defer_mask.sum() > 0:
                correct += (preds[non_defer_mask] == labels[non_defer_mask]).sum().item()
                total += non_defer_mask.sum().item()
                
                for i in range(batch_size):
                    if not defer_mask[i]:
                        all_preds.append(preds[i].item())
                        all_labels.append(labels[i].item())
            
            # Track deferred samples
            deferred += defer_mask.sum().item()
            
            for i in range(batch_size):
                if defer_mask[i]:
                    deferred_indices.append((batch_idx * loader.batch_size + i, labels[i].item()))
    
    # Calculate metrics
    accuracy = 100 * correct / total if total > 0 else 0.0
    deferral_rate = 100 * deferred / (total + deferred) if (total + deferred) > 0 else 0.0
    risk = 100 * ((total - correct) + deferred) / (total + deferred) if (total + deferred) > 0 else 0.0
    
    return accuracy, deferral_rate, risk, all_preds, all_labels, deferred_indices

# ─── Multi-Head CNN Evaluation ───

print("\n📊 Evaluating Multi-Head CNN...")

# 1. Clean accuracy (majority vote)
clean_acc, _, _ = evaluate_multi_head_accuracy(multi_head_model, eval_loader)
print(f"  ✅ Clean accuracy: {clean_acc:.2f}%")

# 2. Adversarial accuracy (PGD attack)
pgd_params = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}
adv_acc, _, _ = evaluate_multi_head_accuracy(
    multi_head_model, eval_loader, 
    attack_type='pgd', attack_params=pgd_params
)
print(f"  ⚔️ Adversarial accuracy (PGD): {adv_acc:.2f}%")

# 3. Evaluation with deferral (clean)
best_threshold = 0.05

clean_acc_def, clean_def_rate, clean_risk, _, _, clean_deferred = evaluate_with_deferral(
    multi_head_model, eval_loader, threshold=best_threshold, attack_type=None
)
print(f"  📤 Clean deferral rate (threshold={best_threshold}): {clean_def_rate:.2f}%")
print(f"  📊 Clean deferred accuracy: {clean_acc_def:.2f}%")
print(f"  ⚠️ Clean risk: {clean_risk:.2f}%")

# 4. Evaluation with deferral (adversarial)
adv_acc_def, adv_def_rate, adv_risk, _, _, adv_deferred = evaluate_with_deferral(
    multi_head_model, eval_loader, threshold=best_threshold,
    attack_type='pgd', attack_params=pgd_params
)
print(f"  📤 Adversarial deferral rate (threshold={best_threshold}): {adv_def_rate:.2f}%")
print(f"  📊 Adversarial deferred accuracy: {adv_acc_def:.2f}%")
print(f"  ⚠️ Adversarial risk: {adv_risk:.2f}%")

print("\n" + "="*55)
print("✅ Multi-Head CNN evaluation complete!")
print("="*55)



🔍 Multi-Head CNN Evaluation

📊 Evaluating Multi-Head CNN...
  ✅ Clean accuracy: 79.50%
  ⚔️ Adversarial accuracy (PGD): 17.12%
  📤 Clean deferral rate (threshold=0.05): 0.00%
  📊 Clean deferred accuracy: 79.49%
  ⚠️ Clean risk: 20.51%
  📤 Adversarial deferral rate (threshold=0.05): 0.00%
  📊 Adversarial deferred accuracy: 17.12%
  ⚠️ Adversarial risk: 82.88%

✅ Multi-Head CNN evaluation complete!


In [ ]:
# -------------------------------------------------------------
# Part F: Signal collection -- confidence, disagreement, evidence,
# boundary distance, all computed the SAME way on calib vs eval,
# clean vs adversarial (PGD). Boundary distance is limited by
# n_boundary_samples (a real subsample, no padding).
# -------------------------------------------------------------

def collect_signals(baseline, multihead, evidential, loader, condition='clean',
                     n_samples=None, n_boundary_samples=300):
    """
    condition: 'clean' or 'adversarial' (PGD against baseline_model).
    n_samples: cap total images processed (None = whole loader).
    n_boundary_samples: how many of those images ALSO get a real
      boundary-distance score (expensive: up to 30 passes each).
      Images beyond this count get boundary=None, dropped from any
      boundary-distance-only or fused-with-boundary evaluation --
      NOT padded with a stale value.
    """
    out = {'confidence': [], 'disagreement': [], 'evidence': [], 'boundary': [],
           'correct_baseline': [], 'correct_multihead': [], 'correct_evidential': []}
    n_seen = 0
    n_boundary_done = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if condition == 'adversarial':
            images = pgd_standard(baseline, images, labels, **PGD_PARAMS)

        with torch.no_grad():
            base_logits = baseline(images)
            base_probs = F.softmax(base_logits, dim=1)
            base_conf, base_pred = base_probs.max(dim=1)

            mean_probs, var_probs = multihead.forward_with_disagreement(images)
            mh_pred = mean_probs.argmax(dim=1)
            disagreement = var_probs.max(dim=1)[0]

            ev_probs, evidence = evidential.forward_with_evidence(images)
            ev_pred = ev_probs.argmax(dim=1)

        for i in range(images.size(0)):
            if n_samples is not None and n_seen >= n_samples:
                break
            out['confidence'].append(base_conf[i].item())
            out['disagreement'].append(disagreement[i].item())
            out['evidence'].append(evidence[i].item())
            out['correct_baseline'].append(int(base_pred[i].item() == labels[i].item()))
            out['correct_multihead'].append(int(mh_pred[i].item() == labels[i].item()))
            out['correct_evidential'].append(int(ev_pred[i].item() == labels[i].item()))

            if n_boundary_done < n_boundary_samples:
                bd = estimate_boundary_distance(baseline, images[i:i+1], labels[i].item(), max_iters=30)
                out['boundary'].append(bd)
                n_boundary_done += 1
            else:
                out['boundary'].append(None)  # explicitly missing, not padded

            n_seen += 1
        if n_samples is not None and n_seen >= n_samples:
            break
        if n_boundary_done % 100 == 0 and n_boundary_done > 0:
            print(f"   ...boundary distance: {n_boundary_done}/{n_boundary_samples}")

    for k in out:
        out[k] = np.array(out[k], dtype=object) if k == 'boundary' else np.array(out[k])
    return out

print("Collecting calibration signals (clean + adversarial)...")
calib_clean = collect_signals(baseline_model, multi_head_model, evidential_model, calib_loader,
                               condition='clean', n_boundary_samples=200)
calib_adv = collect_signals(baseline_model, multi_head_model, evidential_model, calib_loader,
                             condition='adversarial', n_boundary_samples=200)

print("Collecting evaluation signals (clean + adversarial)...")
eval_clean = collect_signals(baseline_model, multi_head_model, evidential_model, eval_loader,
                              condition='clean', n_boundary_samples=300)
eval_adv = collect_signals(baseline_model, multi_head_model, evidential_model, eval_loader,
                            condition='adversarial', n_boundary_samples=300)


✅ Loaded evidential model from checkpoint


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part F: Evidential CNN Evaluation Report 
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("🔬 Evidential CNN Evaluation")
print("="*55)

# ─── Ensure model is loaded ───

if 'evidential_model' not in locals():
    print("🔄 Loading evidential model...")
    
    # Define EvidentialCNN class if not already defined
    class EvidentialCNN(nn.Module):
        """CNN with Dirichlet evidential output."""
        def __init__(self, backbone, num_classes=10):
            super().__init__()
            self.backbone = backbone
            self.fc_alpha = nn.Linear(256, num_classes)

        def forward(self, x):
            features = self.backbone(x)
            alpha = F.softplus(self.fc_alpha(features)) + 1.0
            return alpha

        def forward_with_evidence(self, x):
            alpha = self.forward(x)
            S = alpha.sum(dim=1, keepdim=True)
            probs = alpha / S
            evidence = S.squeeze(1)
            return probs, evidence
    
    evidential_model = EvidentialCNN(BackboneCNN()).to(DEVICE)
    
    if os.path.exists('checkpoint_evidential.pth'):
        checkpoint = torch.load('checkpoint_evidential.pth', map_location=DEVICE)
        evidential_model.load_state_dict(checkpoint['model_state_dict'])
        evidential_model.eval()
        print("✅ Loaded evidential model from checkpoint")
    else:
        print("❌ No checkpoint found. Please train the model first.")
        raise SystemExit("Evidential model not available")

# ─── Attack functions for evidential models ───

def fgsm_attack_evidential(model, images, labels, epsilon=0.03):
    """FGSM attack for evidential models with proper gradient handling."""
    model.eval()
    
    # Clone and require gradients
    images_adv = images.clone().detach().requires_grad_(True)
    
    # Forward pass
    alpha = model(images_adv)
    criterion = nn.CrossEntropyLoss()
    loss = criterion(alpha, labels)
    
    # Backward pass
    model.zero_grad()
    loss.backward()
    
    # Create perturbation
    with torch.no_grad():
        grad_sign = images_adv.grad.sign()
        images_adv = images_adv + epsilon * grad_sign
        images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def pgd_attack_evidential(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    """PGD attack for evidential models with proper gradient handling."""
    model.eval()
    
    # Start from clean images
    images_adv = images.clone().detach()
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(num_steps):
        # Require gradients
        images_adv.requires_grad_(True)
        
        # Forward pass
        alpha = model(images_adv)
        loss = criterion(alpha, labels)
        
        # Backward pass
        model.zero_grad()
        loss.backward()
        
        # Update perturbation
        with torch.no_grad():
            grad_sign = images_adv.grad.sign()
            images_adv = images_adv + step_size * grad_sign
            
            # Project back to epsilon ball
            perturbation = images_adv - images
            perturbation = torch.clamp(perturbation, -epsilon, epsilon)
            images_adv = images + perturbation
            images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

# ─── Evaluation functions for evidential models ───

def evaluate_evidential_accuracy(model, loader, attack_type=None, attack_params=None):
    """Evaluate evidential model accuracy."""
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_evidence = []
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_evidential(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_evidential(model, images, labels, **attack_params)
        
        # Get predictions and evidence
        with torch.no_grad():
            probs, evidence = model.forward_with_evidence(images)
            preds = probs.argmax(dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_evidence.extend(evidence.cpu().numpy())
    
    return 100 * correct / total, all_preds, all_labels, all_evidence

def evaluate_evidential_with_deferral(model, loader, threshold, attack_type=None, attack_params=None):
    """
    Evaluate evidential model with deferral based on evidence threshold.
    Returns: (accuracy, deferral_rate, risk, predictions, labels, evidence_values, deferred_indices)
    """
    model.eval()
    correct = 0
    total = 0
    deferred = 0
    all_preds = []
    all_labels = []
    all_evidence = []
    deferred_indices = []
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        batch_size = images.size(0)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_evidential(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_evidential(model, images, labels, **attack_params)
        
        # Get predictions and evidence
        with torch.no_grad():
            probs, evidence = model.forward_with_evidence(images)
            preds = probs.argmax(dim=1)
            
            # Defer based on low evidence
            defer_mask = evidence < threshold
            
            # For non-deferred samples, check accuracy
            non_defer_mask = ~defer_mask
            if non_defer_mask.sum() > 0:
                correct += (preds[non_defer_mask] == labels[non_defer_mask]).sum().item()
                total += non_defer_mask.sum().item()
                
                for i in range(batch_size):
                    if not defer_mask[i]:
                        all_preds.append(preds[i].item())
                        all_labels.append(labels[i].item())
                        all_evidence.append(float(evidence[i].item()))  # Convert to float
            
            # Track deferred samples
            deferred += defer_mask.sum().item()
            
            for i in range(batch_size):
                if defer_mask[i]:
                    deferred_indices.append((batch_idx * loader.batch_size + i, labels[i].item(), float(evidence[i].item())))
    
    # Calculate metrics
    accuracy = 100 * correct / total if total > 0 else 0.0
    deferral_rate = 100 * deferred / (total + deferred) if (total + deferred) > 0 else 0.0
    risk = 100 * ((total - correct) + deferred) / (total + deferred) if (total + deferred) > 0 else 0.0
    
    return accuracy, deferral_rate, risk, all_preds, all_labels, all_evidence, deferred_indices

# ─── Evidential CNN Evaluation ───

print("\n📊 Evaluating Evidential CNN...")

# 1. Clean accuracy
clean_acc_evid, _, _, clean_evidence = evaluate_evidential_accuracy(evidential_model, eval_loader)
print(f"  ✅ Clean accuracy: {clean_acc_evid:.2f}%")

# 2. Adversarial accuracy (PGD)
pgd_params = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}
adv_acc_evid, _, _, adv_evidence = evaluate_evidential_accuracy(
    evidential_model, eval_loader, 
    attack_type='pgd', attack_params=pgd_params
)
print(f"  ⚔️ Adversarial accuracy (PGD): {adv_acc_evid:.2f}%")

# 3. Evidence statistics - convert to float for JSON serialization
mean_evidence_clean = float(np.mean(clean_evidence)) if clean_evidence else 0.0
std_evidence_clean = float(np.std(clean_evidence)) if clean_evidence else 0.0
mean_evidence_adv = float(np.mean(adv_evidence)) if adv_evidence else 0.0
std_evidence_adv = float(np.std(adv_evidence)) if adv_evidence else 0.0

print(f"\n📊 Evidence Statistics:")
print(f"  Clean - Mean: {mean_evidence_clean:.3f}, Std: {std_evidence_clean:.3f}")
print(f"  Adversarial - Mean: {mean_evidence_adv:.3f}, Std: {std_evidence_adv:.3f}")

# 4. Evaluation with deferral - Clean
# Use a threshold based on evidence distribution (e.g., 10th percentile)
clean_threshold = np.percentile(clean_evidence, 10) if clean_evidence else 10.0
print(f"\n📊 Using deferral threshold: {clean_threshold:.3f} (10th percentile of clean evidence)")

clean_acc_def_evid, clean_def_rate_evid, clean_risk_evid, _, _, clean_evidence_def, clean_deferred_evid = evaluate_evidential_with_deferral(
    evidential_model, eval_loader, threshold=clean_threshold, attack_type=None
)
print(f"\n📤 Clean deferral rate (threshold={clean_threshold:.3f}): {clean_def_rate_evid:.2f}%")
print(f"  📊 Clean deferred accuracy: {clean_acc_def_evid:.2f}%")
print(f"  ⚠️ Clean risk: {clean_risk_evid:.2f}%")

# 5. Evaluation with deferral - Adversarial
adv_threshold = np.percentile(adv_evidence, 10) if adv_evidence else 10.0
print(f"\n📊 Using adversarial deferral threshold: {adv_threshold:.3f} (10th percentile of adversarial evidence)")

adv_acc_def_evid, adv_def_rate_evid, adv_risk_evid, _, _, _, adv_deferred_evid = evaluate_evidential_with_deferral(
    evidential_model, eval_loader, threshold=adv_threshold,
    attack_type='pgd', attack_params=pgd_params
)
print(f"\n📤 Adversarial deferral rate (threshold={adv_threshold:.3f}): {adv_def_rate_evid:.2f}%")
print(f"  📊 Adversarial deferred accuracy: {adv_acc_def_evid:.2f}%")
print(f"  ⚠️ Adversarial risk: {adv_risk_evid:.2f}%")

# ─── Detailed Evidential Report ───

print("\n" + "="*55)
print("📈 EVIDENTIAL CNN EVALUATION REPORT")
print("="*55)

report_data_evid = [
    ['Metric', 'Value', 'Description'],
    ['Clean Accuracy', f'{clean_acc_evid:.2f}%', 'Accuracy on clean test data'],
    ['Adversarial Accuracy', f'{adv_acc_evid:.2f}%', 'PGD attack (ε=0.03) accuracy'],
    ['Clean Evidence Mean', f'{mean_evidence_clean:.3f}', 'Mean total evidence for clean samples'],
    ['Adversarial Evidence Mean', f'{mean_evidence_adv:.3f}', 'Mean total evidence under attack'],
    ['Clean Deferral Rate', f'{clean_def_rate_evid:.2f}%', f'Deferred samples (threshold={clean_threshold:.3f})'],
    ['Clean Deferred Accuracy', f'{clean_acc_def_evid:.2f}%', 'Accuracy on non-deferred clean samples'],
    ['Clean Risk', f'{clean_risk_evid:.2f}%', 'Conservative risk (deferrals = errors)'],
    ['Adversarial Deferral Rate', f'{adv_def_rate_evid:.2f}%', f'Deferred samples under PGD (threshold={adv_threshold:.3f})'],
    ['Adversarial Deferred Accuracy', f'{adv_acc_def_evid:.2f}%', 'Accuracy on non-deferred adversarial samples'],
    ['Adversarial Risk', f'{adv_risk_evid:.2f}%', 'Conservative risk under PGD attack'],
    ['Deferred Clean Samples', f'{len(clean_deferred_evid)}', 'Number of samples deferred on clean data'],
    ['Deferred Adversarial Samples', f'{len(adv_deferred_evid)}', 'Number of samples deferred under attack'],
]

for row in report_data_evid:
    print(f"{row[0]:<30} {row[1]:<15} {row[2]}")

print("="*55)

# ─── Evidence Distribution Analysis ───

if clean_evidence and adv_evidence:
    print("\n📊 EVIDENCE DISTRIBUTION ANALYSIS:")
    # Convert to float for printing
    clean_25 = float(np.percentile(clean_evidence, 25))
    clean_50 = float(np.percentile(clean_evidence, 50))
    clean_75 = float(np.percentile(clean_evidence, 75))
    adv_25 = float(np.percentile(adv_evidence, 25))
    adv_50 = float(np.percentile(adv_evidence, 50))
    adv_75 = float(np.percentile(adv_evidence, 75))
    
    print(f"  • Clean evidence - 25th percentile: {clean_25:.3f}")
    print(f"  • Clean evidence - 50th percentile: {clean_50:.3f}")
    print(f"  • Clean evidence - 75th percentile: {clean_75:.3f}")
    print(f"  • Adversarial evidence - 25th percentile: {adv_25:.3f}")
    print(f"  • Adversarial evidence - 50th percentile: {adv_50:.3f}")
    print(f"  • Adversarial evidence - 75th percentile: {adv_75:.3f}")

# ─── Save Evidential Results (Fixed JSON serialization) ───

import json

# Convert all numpy values to Python floats for JSON serialization
evidential_results = {
    'clean_accuracy': float(clean_acc_evid),
    'adversarial_accuracy': float(adv_acc_evid),
    'clean_evidence_mean': float(mean_evidence_clean),
    'adversarial_evidence_mean': float(mean_evidence_adv),
    'clean_evidence_std': float(std_evidence_clean),
    'adversarial_evidence_std': float(std_evidence_adv),
    'clean_deferral_rate': float(clean_def_rate_evid),
    'clean_deferred_accuracy': float(clean_acc_def_evid),
    'clean_risk': float(clean_risk_evid),
    'adversarial_deferral_rate': float(adv_def_rate_evid),
    'adversarial_deferred_accuracy': float(adv_acc_def_evid),
    'adversarial_risk': float(adv_risk_evid),
    'clean_threshold': float(clean_threshold),
    'adversarial_threshold': float(adv_threshold),
    'num_deferred_clean': int(len(clean_deferred_evid)),
    'num_deferred_adversarial': int(len(adv_deferred_evid)),
}

if clean_evidence and adv_evidence:
    evidential_results['evidence_percentiles_clean'] = {
        '25th': float(np.percentile(clean_evidence, 25)),
        '50th': float(np.percentile(clean_evidence, 50)),
        '75th': float(np.percentile(clean_evidence, 75))
    }
    evidential_results['evidence_percentiles_adv'] = {
        '25th': float(np.percentile(adv_evidence, 25)),
        '50th': float(np.percentile(adv_evidence, 50)),
        '75th': float(np.percentile(adv_evidence, 75))
    }

with open('evidential_results.json', 'w') as f:
    json.dump(evidential_results, f, indent=2)
print("\n💾 Results saved to 'evidential_results.json'")

print("\n" + "="*55)
print("✅ Evidential CNN evaluation complete!")
print("="*55)



🔬 Evidential CNN Evaluation

📊 Evaluating Evidential CNN...
  ✅ Clean accuracy: 76.95%
  ⚔️ Adversarial accuracy (PGD): 22.50%

📊 Evidence Statistics:
  Clean - Mean: 437.328, Std: 287.886
  Adversarial - Mean: 124.602, Std: 84.428

📊 Using deferral threshold: 136.739 (10th percentile of clean evidence)

📤 Clean deferral rate (threshold=136.739): 10.00%
  📊 Clean deferred accuracy: 79.81%
  ⚠️ Clean risk: 28.18%

📊 Using adversarial deferral threshold: 39.838 (10th percentile of adversarial evidence)

📤 Adversarial deferral rate (threshold=39.838): 10.00%
  📊 Adversarial deferred accuracy: 24.22%
  ⚠️ Adversarial risk: 78.20%

📈 EVIDENTIAL CNN EVALUATION REPORT
Metric                         Value           Description
Clean Accuracy                 76.95%          Accuracy on clean test data
Adversarial Accuracy           22.50%          PGD attack (ε=0.03) accuracy
Clean Evidence Mean            437.328         Mean total evidence for clean samples
Adversarial Evidence Mean      124

In [ ]:
# -------------------------------------------------------------
# Part G: Calibrate thresholds ON CALIB DATA ONLY
# -------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.7  # matches Essays #4/#5's convention, kept fixed for comparability
disagreement_threshold = float(np.percentile(calib_clean['disagreement'], 75))
evidence_threshold = float(np.percentile(calib_clean['evidence'], 10))
boundary_vals_calib = np.array([b for b in calib_clean['boundary'] if b is not None])
boundary_threshold = float(np.percentile(boundary_vals_calib, 25))

print(f"\nThresholds calibrated on {len(calib_dataset)}-image calibration set:")
print(f"   Confidence:   {CONFIDENCE_THRESHOLD} (fixed, matches Essays 4/5)")
print(f"   Disagreement: {disagreement_threshold:.6f} (75th pct of clean calib)")
print(f"   Evidence:     {evidence_threshold:.4f} (10th pct of clean calib)")
print(f"   Boundary:     {boundary_threshold:.4f} (25th pct of clean calib)")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Part H: Head-to-Head Comparison--Report 
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("🔍 Head-to-Head Comparison: Baseline vs Multi-Head vs Evidential")
print("="*55)

# ─── Ensure models are loaded ───

# Baseline model (from Part D)
if 'baseline_model' not in locals():
    print("🔄 Loading baseline model...")
    baseline_model = StandardCNN().to(DEVICE)
    if os.path.exists('checkpoint_baseline_cifar10.pth'):
        checkpoint = torch.load('checkpoint_baseline_cifar10.pth', map_location=DEVICE)
        baseline_model.load_state_dict(checkpoint['model_state_dict'])
        baseline_model.eval()
        print("✅ Loaded baseline model from checkpoint")
    else:
        print("❌ No baseline checkpoint found.")
        raise SystemExit("Baseline model not available")

# Multi-head model (from Part E)
if 'multi_head_model' not in locals():
    print("🔄 Loading multi-head model...")
    multi_head_model = MultiHeadCNN(BackboneCNN(), num_heads=5).to(DEVICE)
    if os.path.exists('checkpoint_multihead.pth'):
        checkpoint = torch.load('checkpoint_multihead.pth', map_location=DEVICE)
        multi_head_model.load_state_dict(checkpoint['model_state_dict'])
        multi_head_model.eval()
        print("✅ Loaded multi-head model from checkpoint")
    else:
        print("❌ No multi-head checkpoint found.")
        raise SystemExit("Multi-head model not available")

# Evidential model (from Part F)
if 'evidential_model' not in locals():
    print("🔄 Loading evidential model...")
    evidential_model = EvidentialCNN(BackboneCNN()).to(DEVICE)
    if os.path.exists('checkpoint_evidential.pth'):
        checkpoint = torch.load('checkpoint_evidential.pth', map_location=DEVICE)
        evidential_model.load_state_dict(checkpoint['model_state_dict'])
        evidential_model.eval()
        print("✅ Loaded evidential model from checkpoint")
    else:
        print("❌ No evidential checkpoint found.")
        raise SystemExit("Evidential model not available")

# ─── Helper functions ───

def evaluate_accuracy_with_attack(model, loader, attack_type=None, attack_params=None):
    """Evaluate model accuracy with optional attack."""
    model.eval()
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_simple(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_simple(model, images, labels, **attack_params)
        
        # Get predictions
        with torch.no_grad():
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return 100 * correct / total, None, None

def fgsm_attack_simple(model, images, labels, epsilon=0.03):
    """Simple FGSM attack for standard models."""
    model.eval()
    images_adv = images.clone().detach().requires_grad_(True)
    
    outputs = model(images_adv)
    criterion = nn.CrossEntropyLoss()
    loss = criterion(outputs, labels)
    
    model.zero_grad()
    loss.backward()
    
    with torch.no_grad():
        grad_sign = images_adv.grad.sign()
        images_adv = images_adv + epsilon * grad_sign
        images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def pgd_attack_simple(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    """Simple PGD attack for standard models."""
    model.eval()
    images_adv = images.clone().detach()
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        outputs = model(images_adv)
        loss = criterion(outputs, labels)
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            grad_sign = images_adv.grad.sign()
            images_adv = images_adv + step_size * grad_sign
            perturbation = images_adv - images
            perturbation = torch.clamp(perturbation, -epsilon, epsilon)
            images_adv = images + perturbation
            images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def evaluate_baseline_with_deferral(model, loader, threshold, attack_type=None, attack_params=None):
    """Evaluate baseline model with deferral based on confidence threshold."""
    model.eval()
    correct = 0
    total = 0
    deferred = 0
    all_preds = []
    all_labels = []
    all_confidences = []
    deferred_indices = []
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        batch_size = images.size(0)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images = fgsm_attack_simple(model, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images = pgd_attack_simple(model, images, labels, **attack_params)
        
        # Get predictions and confidence
        with torch.no_grad():
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            preds = probs.argmax(dim=1)
            confidence = probs.max(dim=1)[0]
            
            # Defer based on low confidence
            defer_mask = confidence < threshold
            
            non_defer_mask = ~defer_mask
            if non_defer_mask.sum() > 0:
                correct += (preds[non_defer_mask] == labels[non_defer_mask]).sum().item()
                total += non_defer_mask.sum().item()
                
                for i in range(batch_size):
                    if not defer_mask[i]:
                        all_preds.append(preds[i].item())
                        all_labels.append(labels[i].item())
                        all_confidences.append(float(confidence[i].item()))
            
            deferred += defer_mask.sum().item()
            
            for i in range(batch_size):
                if defer_mask[i]:
                    deferred_indices.append((batch_idx * loader.batch_size + i, labels[i].item(), float(confidence[i].item())))
    
    accuracy = 100 * correct / total if total > 0 else 0.0
    deferral_rate = 100 * deferred / (total + deferred) if (total + deferred) > 0 else 0.0
    risk = 100 * ((total - correct) + deferred) / (total + deferred) if (total + deferred) > 0 else 0.0
    
    return accuracy, deferral_rate, risk, all_preds, all_labels, all_confidences, deferred_indices

# ─── Collect metrics for all models ───

print("\n📊 Collecting metrics for all models...")

# Baseline model
pgd_params = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}

# Baseline clean accuracy (already defined)
baseline_clean_acc = clean_acc

# Baseline adversarial accuracy
baseline_adv_acc, _, _ = evaluate_accuracy_with_attack(
    baseline_model, eval_loader, 
    attack_type='pgd', attack_params=pgd_params
)
print(f"  Baseline clean: {baseline_clean_acc:.2f}%, adversarial: {baseline_adv_acc:.2f}%")

# Multi-head model (from Part E)
# Assuming these variables exist from Part E
try:
    multi_head_clean_acc = clean_acc  # From Part E
    multi_head_adv_acc = adv_acc  # From Part E
except NameError:
    # If not defined, compute them
    print("  Computing multi-head metrics...")
    multi_head_clean_acc, _, _ = evaluate_multi_head_accuracy(multi_head_model, eval_loader)
    multi_head_adv_acc, _, _ = evaluate_multi_head_accuracy(
        multi_head_model, eval_loader, 
        attack_type='pgd', attack_params=pgd_params
    )

# Evidential model (from Part F)
try:
    evidential_clean_acc = clean_acc_evid  # From Part F
    evidential_adv_acc = adv_acc_evid  # From Part F
except NameError:
    # If not defined, compute them
    print("  Computing evidential metrics...")
    evidential_clean_acc, _, _, _ = evaluate_evidential_accuracy(evidential_model, eval_loader)
    evidential_adv_acc, _, _, _ = evaluate_evidential_accuracy(
        evidential_model, eval_loader, 
        attack_type='pgd', attack_params=pgd_params
    )

# ─── Evaluate with deferral for all models ───

print("\n📊 Evaluating with deferral...")

# Baseline with deferral (confidence threshold)
baseline_clean_def_acc, baseline_clean_def_rate, baseline_clean_risk, _, _, _, baseline_clean_deferred = evaluate_baseline_with_deferral(
    baseline_model, eval_loader, threshold=0.7, attack_type=None
)

baseline_adv_def_acc, baseline_adv_def_rate, baseline_adv_risk, _, _, _, baseline_adv_deferred = evaluate_baseline_with_deferral(
    baseline_model, eval_loader, threshold=0.7, attack_type='pgd', attack_params=pgd_params
)

# Multi-head with deferral (using previously computed values from Part E)
try:
    multi_clean_def_acc = clean_acc_def
    multi_clean_def_rate = clean_def_rate
    multi_clean_risk = clean_risk
    multi_adv_def_acc = adv_acc_def
    multi_adv_def_rate = adv_def_rate
    multi_adv_risk = adv_risk
except NameError:
    # If not defined, compute them
    print("  Computing multi-head deferral metrics...")
    multi_clean_def_acc, multi_clean_def_rate, multi_clean_risk, _, _, _ = evaluate_with_deferral(
        multi_head_model, eval_loader, threshold=0.05, attack_type=None
    )
    multi_adv_def_acc, multi_adv_def_rate, multi_adv_risk, _, _, _ = evaluate_with_deferral(
        multi_head_model, eval_loader, threshold=0.05,
        attack_type='pgd', attack_params=pgd_params
    )

# Evidential with deferral (using previously computed values from Part F)
try:
    evid_clean_def_acc = clean_acc_def_evid
    evid_clean_def_rate = clean_def_rate_evid
    evid_clean_risk = clean_risk_evid
    evid_adv_def_acc = adv_acc_def_evid
    evid_adv_def_rate = adv_def_rate_evid
    evid_adv_risk = adv_risk_evid
except NameError:
    # If not defined, compute them
    print("  Computing evidential deferral metrics...")
    clean_threshold = np.percentile(clean_evidence, 10) if 'clean_evidence' in locals() else 3.0
    evid_clean_def_acc, evid_clean_def_rate, evid_clean_risk, _, _, _, _ = evaluate_evidential_with_deferral(
        evidential_model, eval_loader, threshold=clean_threshold, attack_type=None
    )
    adv_threshold = np.percentile(adv_evidence, 10) if 'adv_evidence' in locals() else 3.0
    evid_adv_def_acc, evid_adv_def_rate, evid_adv_risk, _, _, _, _ = evaluate_evidential_with_deferral(
        evidential_model, eval_loader, threshold=adv_threshold,
        attack_type='pgd', attack_params=pgd_params
    )

# ─── Comprehensive Comparison Table ───

print("\n" + "="*55)
print("📊 HEAD-TO-HEAD COMPARISON SUMMARY")
print("="*55)

print("\n┌─────────────────────┬──────────────┬──────────────┬──────────────┐")
print("│ Metric              │   Baseline   │ Multi-Head   │ Evidential   │")
print("├─────────────────────┼──────────────┼──────────────┼──────────────┤")

print(f"│ Clean Acc           │ {baseline_clean_acc:>11.2f}% │ {multi_head_clean_acc:>11.2f}% │ {evidential_clean_acc:>11.2f}% │")
print(f"│ Adv Acc             │ {baseline_adv_acc:>11.2f}% │ {multi_head_adv_acc:>11.2f}% │ {evidential_adv_acc:>11.2f}% │")
print(f"│ Clean Def Rate      │ {baseline_clean_def_rate:>11.2f}% │ {multi_clean_def_rate:>11.2f}% │ {evid_clean_def_rate:>11.2f}% │")
print(f"│ Clean Risk          │ {baseline_clean_risk:>11.2f}% │ {multi_clean_risk:>11.2f}% │ {evid_clean_risk:>11.2f}% │")
print(f"│ Adv Def Rate        │ {baseline_adv_def_rate:>11.2f}% │ {multi_adv_def_rate:>11.2f}% │ {evid_adv_def_rate:>11.2f}% │")
print(f"│ Adv Risk            │ {baseline_adv_risk:>11.2f}% │ {multi_adv_risk:>11.2f}% │ {evid_adv_risk:>11.2f}% │")

print("└─────────────────────┴──────────────┴──────────────┴──────────────┘")

# ─── Detailed Comparison Report ───

print("\n" + "="*55)
print("📈 DETAILED HEAD-TO-HEAD COMPARISON")
print("="*55)

# Calculate improvements
multi_clean_improvement = multi_head_clean_acc - baseline_clean_acc
multi_adv_improvement = multi_head_adv_acc - baseline_adv_acc
evid_clean_improvement = evidential_clean_acc - baseline_clean_acc
evid_adv_improvement = evidential_adv_acc - baseline_adv_acc

print(f"\n🎯 Accuracy Improvements over Baseline:")
print(f"  • Multi-Head CNN:")
print(f"    - Clean: {multi_clean_improvement:+.2f}%")
print(f"    - Adversarial: {multi_adv_improvement:+.2f}%")
print(f"  • Evidential CNN:")
print(f"    - Clean: {evid_clean_improvement:+.2f}%")
print(f"    - Adversarial: {evid_adv_improvement:+.2f}%")

# Deferral effectiveness
print(f"\n📤 Deferral Effectiveness (Risk Reduction):")
print(f"  • Multi-Head CNN:")
print(f"    - Clean risk reduction vs baseline: {baseline_clean_risk - multi_clean_risk:+.2f}%")
print(f"    - Adversarial risk reduction vs baseline: {baseline_adv_risk - multi_adv_risk:+.2f}%")
print(f"  • Evidential CNN:")
print(f"    - Clean risk reduction vs baseline: {baseline_clean_risk - evid_clean_risk:+.2f}%")
print(f"    - Adversarial risk reduction vs baseline: {baseline_adv_risk - evid_adv_risk:+.2f}%")

# Robustness comparison
print(f"\n🛡️ Robustness (Clean - Adversarial Gap):")
print(f"  • Baseline: {baseline_clean_acc - baseline_adv_acc:+.2f}%")
print(f"  • Multi-Head CNN: {multi_head_clean_acc - multi_head_adv_acc:+.2f}%")
print(f"  • Evidential CNN: {evidential_clean_acc - evidential_adv_acc:+.2f}%")

# ─── Save Comparison Results ───

import json

# Convert all values to Python native types for JSON serialization
comparison_results = {
    'baseline': {
        'clean_accuracy': float(baseline_clean_acc),
        'adversarial_accuracy': float(baseline_adv_acc),
        'clean_deferral_rate': float(baseline_clean_def_rate),
        'clean_risk': float(baseline_clean_risk),
        'adversarial_deferral_rate': float(baseline_adv_def_rate),
        'adversarial_risk': float(baseline_adv_risk)
    },
    'multi_head': {
        'clean_accuracy': float(multi_head_clean_acc),
        'adversarial_accuracy': float(multi_head_adv_acc),
        'clean_deferral_rate': float(multi_clean_def_rate),
        'clean_risk': float(multi_clean_risk),
        'adversarial_deferral_rate': float(multi_adv_def_rate),
        'adversarial_risk': float(multi_adv_risk)
    },
    'evidential': {
        'clean_accuracy': float(evidential_clean_acc),
        'adversarial_accuracy': float(evidential_adv_acc),
        'clean_deferral_rate': float(evid_clean_def_rate),
        'clean_risk': float(evid_clean_risk),
        'adversarial_deferral_rate': float(evid_adv_def_rate),
        'adversarial_risk': float(evid_adv_risk)
    },
    'improvements': {
        'multi_head_clean': float(multi_clean_improvement),
        'multi_head_adversarial': float(multi_adv_improvement),
        'evidential_clean': float(evid_clean_improvement),
        'evidential_adversarial': float(evid_adv_improvement)
    }
}

with open('comparison_results.json', 'w') as f:
    json.dump(comparison_results, f, indent=2)
print("\n💾 Comparison results saved to 'comparison_results.json'")

print("\n" + "="*55)
print("✅ Head-to-Head Comparison complete!")
print("="*55)


🔍 Head-to-Head Comparison: Baseline vs Multi-Head vs Evidential


NameError: name 'evaluate_accuracy_with_attack' is not defined

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part I: Fused Gate Evaluation 
# ─────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("🔀 Fused Gate Evaluation")
print("="*55)

# ─── Helper Functions ───

def compute_boundary_distance(model, image, label, max_iters=30):
    """
    Estimate boundary distance using binary search.
    FIXED: Proper gradient handling.
    """
    model.eval()
    
    # Clone and require gradients
    image = image.clone().detach().requires_grad_(True)
    
    # Compute gradient direction
    output = model(image)
    loss = nn.CrossEntropyLoss()(output, torch.tensor([label], device=DEVICE))
    model.zero_grad()
    loss.backward()
    
    # Get gradient sign
    grad = image.grad.data.sign()
    
    # Binary search for boundary
    low, high = 0.0, 1.0
    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = image + mid * grad
        perturbed = torch.clamp(perturbed, 0, 1)
        
        with torch.no_grad():
            pred = model(perturbed).argmax().item()
        
        if pred != label:
            high = mid
        else:
            low = mid
    
    return (low + high) / 2

def get_confidence(model, images):
    """Get prediction confidence."""
    model.eval()
    with torch.no_grad():
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        confidence = probs.max(dim=1)[0]
    return confidence

def get_disagreement(multi_head_model, images):
    """Get disagreement across heads."""
    multi_head_model.eval()
    with torch.no_grad():
        mean_probs, var_probs = multi_head_model.forward_with_disagreement(images)
        disagreement = var_probs.max(dim=1)[0]
    return disagreement

def get_evidence(evidential_model, images):
    """Get total evidence."""
    evidential_model.eval()
    with torch.no_grad():
        _, evidence = evidential_model.forward_with_evidence(images)
    return evidence

def fgsm_attack_fused(model, images, labels, epsilon=0.03):
    """FGSM attack for fused gate evaluation."""
    model.eval()
    images_adv = images.clone().detach().requires_grad_(True)
    
    outputs = model(images_adv)
    criterion = nn.CrossEntropyLoss()
    loss = criterion(outputs, labels)
    
    model.zero_grad()
    loss.backward()
    
    with torch.no_grad():
        grad_sign = images_adv.grad.sign()
        images_adv = images_adv + epsilon * grad_sign
        images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

def pgd_attack_fused(model, images, labels, epsilon=0.03, step_size=0.007, num_steps=40):
    """PGD attack for fused gate evaluation."""
    model.eval()
    images_adv = images.clone().detach()
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(num_steps):
        images_adv.requires_grad_(True)
        outputs = model(images_adv)
        loss = criterion(outputs, labels)
        
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            grad_sign = images_adv.grad.sign()
            images_adv = images_adv + step_size * grad_sign
            perturbation = images_adv - images
            perturbation = torch.clamp(perturbation, -epsilon, epsilon)
            images_adv = images + perturbation
            images_adv = torch.clamp(images_adv, 0, 1)
    
    return images_adv.detach()

# ─── Main Fused Gate Evaluation Function ───

def evaluate_fused_gate(model_baseline, model_multi, model_evidential, loader, 
                        attack_type=None, attack_params=None,
                        w_confidence=0.3, w_disagreement=0.3, w_evidence=0.2, w_boundary=0.2,
                        threshold=0.5, max_boundary_samples=100):
    """
    Evaluate fused gate using all three models.
    FIXED: Limited boundary distance computation for speed.
    """
    model_baseline.eval()
    model_multi.eval()
    model_evidential.eval()
    
    correct = 0
    total = 0
    deferred = 0
    all_preds = []
    all_labels = []
    all_scores = []
    deferred_indices = []
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        batch_size = images.size(0)
        
        # Apply attack if specified
        if attack_type is not None:
            if attack_type == 'fgsm':
                images_attacked = fgsm_attack_fused(model_baseline, images, labels, **attack_params)
            elif attack_type == 'pgd':
                images_attacked = pgd_attack_fused(model_baseline, images, labels, **attack_params)
        else:
            images_attacked = images
        
        # Compute signals
        # 1. Confidence from baseline
        confidence = get_confidence(model_baseline, images_attacked)
        
        # 2. Disagreement from multi-head
        disagreement = get_disagreement(model_multi, images_attacked)
        
        # 3. Evidence from evidential
        evidence = get_evidence(model_evidential, images_attacked)
        
        # 4. Boundary distance (compute for a subset or skip if too slow)
        # For efficiency, we'll use a simplified boundary estimation
        boundary_distances = []
        use_boundary = True
        
        # Only compute boundary for a subset if batch is large
        sample_indices = range(min(batch_size, max_boundary_samples))
        
        for i in sample_indices:
            try:
                bd = compute_boundary_distance(model_baseline, images_attacked[i:i+1], labels[i].item())
                boundary_distances.append(bd)
            except:
                # If boundary computation fails, use a default value
                boundary_distances.append(0.5)
        
        # Pad if we computed fewer samples
        if len(boundary_distances) < batch_size:
            # Repeat the last value or use default
            while len(boundary_distances) < batch_size:
                boundary_distances.append(boundary_distances[-1] if boundary_distances else 0.5)
        
        boundary_distances = torch.tensor(boundary_distances, device=DEVICE)
        
        # Fused decision
        defer_mask = torch.zeros(batch_size, dtype=torch.bool, device=DEVICE)
        scores = []
        
        for i in range(batch_size):
            # Normalize evidence (cap at 10)
            norm_evidence = min(evidence[i].item() / 10.0, 1.0)
            # Normalize boundary distance (1 - distance, so smaller distance = higher score)
            norm_boundary = 1 - min(boundary_distances[i].item(), 1.0)
            
            # Combine scores
            score = (w_confidence * confidence[i].item() + 
                    w_disagreement * disagreement[i].item() + 
                    w_evidence * norm_evidence + 
                    w_boundary * norm_boundary)
            scores.append(score)
            
            if score < threshold:
                defer_mask[i] = True
        
        # Get prediction from baseline
        with torch.no_grad():
            outputs = model_baseline(images_attacked)
            preds = outputs.argmax(dim=1)
        
        # Evaluate non-deferred samples
        non_defer_mask = ~defer_mask
        if non_defer_mask.sum() > 0:
            correct += (preds[non_defer_mask] == labels[non_defer_mask]).sum().item()
            total += non_defer_mask.sum().item()
            
            for i in range(batch_size):
                if not defer_mask[i]:
                    all_preds.append(preds[i].item())
                    all_labels.append(labels[i].item())
                    all_scores.append(scores[i])
        
        deferred += defer_mask.sum().item()
        for i in range(batch_size):
            if defer_mask[i]:
                deferred_indices.append((batch_idx * loader.batch_size + i, labels[i].item()))
    
    accuracy = 100 * correct / total if total > 0 else 0.0
    deferral_rate = 100 * deferred / (total + deferred) if (total + deferred) > 0 else 0.0
    risk = 100 * ((total - correct) + deferred) / (total + deferred) if (total + deferred) > 0 else 0.0
    
    return accuracy, deferral_rate, risk, all_preds, all_labels, all_scores, deferred_indices

# ─── Fused Gate Evaluation ───

print("\n📊 Evaluating Fused Gate...")

# Define weights (these can be tuned)
weights = {
    'w_confidence': 0.3,
    'w_disagreement': 0.3,
    'w_evidence': 0.2,
    'w_boundary': 0.2
}
fused_threshold = 0.4

# Clean evaluation
print("  Running clean evaluation...")
clean_acc_fused, clean_def_fused, clean_risk_fused, _, _, _, clean_deferred_fused = evaluate_fused_gate(
    baseline_model, multi_head_model, evidential_model, eval_loader,
    attack_type=None, attack_params=None,
    w_confidence=weights['w_confidence'],
    w_disagreement=weights['w_disagreement'],
    w_evidence=weights['w_evidence'],
    w_boundary=weights['w_boundary'],
    threshold=fused_threshold,
    max_boundary_samples=50  # Limit boundary computation
)

print(f"\n📊 Clean Results:")
print(f"  ✅ Accuracy: {clean_acc_fused:.2f}%")
print(f"  📤 Deferral rate: {clean_def_fused:.2f}%")
print(f"  ⚠️ Risk: {clean_risk_fused:.2f}%")

# Adversarial evaluation
print("\n  Running adversarial evaluation...")
adv_acc_fused, adv_def_fused, adv_risk_fused, _, _, _, adv_deferred_fused = evaluate_fused_gate(
    baseline_model, multi_head_model, evidential_model, eval_loader,
    attack_type='pgd', attack_params=pgd_params,
    w_confidence=weights['w_confidence'],
    w_disagreement=weights['w_disagreement'],
    w_evidence=weights['w_evidence'],
    w_boundary=weights['w_boundary'],
    threshold=fused_threshold,
    max_boundary_samples=50  # Limit boundary computation
)

print(f"\n📊 Adversarial Results:")
print(f"  ✅ Accuracy: {adv_acc_fused:.2f}%")
print(f"  📤 Deferral rate: {adv_def_fused:.2f}%")
print(f"  ⚠️ Risk: {adv_risk_fused:.2f}%")

# ─── Simple Weight Sensitivity Analysis (Reduced) ───

print("\n" + "="*55)
print("⚖️ WEIGHT SENSITIVITY ANALYSIS (SIMPLIFIED)")
print("="*55)

# Reduced number of configurations for speed
weight_configs = [
    {'w_confidence': 0.4, 'w_disagreement': 0.3, 'w_evidence': 0.2, 'w_boundary': 0.1},
    {'w_confidence': 0.3, 'w_disagreement': 0.4, 'w_evidence': 0.2, 'w_boundary': 0.1},
    {'w_confidence': 0.25, 'w_disagreement': 0.25, 'w_evidence': 0.25, 'w_boundary': 0.25},
]

print("\nConfig | Clean Acc | Clean Def | Clean Risk | Adv Acc | Adv Def | Adv Risk")
print("-" * 90)

for idx, w in enumerate(weight_configs):
    print(f"  Running config {idx+1}...")
    clean_acc_t, clean_def_t, clean_risk_t, _, _, _, _ = evaluate_fused_gate(
        baseline_model, multi_head_model, evidential_model, eval_loader,
        attack_type=None, attack_params=None,
        **w, threshold=fused_threshold,
        max_boundary_samples=30  # Even fewer for sensitivity analysis
    )
    adv_acc_t, adv_def_t, adv_risk_t, _, _, _, _ = evaluate_fused_gate(
        baseline_model, multi_head_model, evidential_model, eval_loader,
        attack_type='pgd', attack_params=pgd_params,
        **w, threshold=fused_threshold,
        max_boundary_samples=30
    )
    print(f"{idx+1:4d}   | {clean_acc_t:>7.2f}%  | {clean_def_t:>7.2f}%  | {clean_risk_t:>7.2f}%  | "
          f"{adv_acc_t:>7.2f}%  | {adv_def_t:>7.2f}%  | {adv_risk_t:>7.2f}%")

# ─── Simple Threshold Sensitivity Analysis ───

print("\n" + "="*55)
print("🎯 THRESHOLD SENSITIVITY ANALYSIS (SIMPLIFIED)")
print("="*55)

thresholds_fused = [0.3, 0.4, 0.5, 0.6]

print("\nThreshold | Clean Acc | Clean Def | Clean Risk | Adv Acc | Adv Def | Adv Risk")
print("-" * 80)

for thresh in thresholds_fused:
    clean_acc_t, clean_def_t, clean_risk_t, _, _, _, _ = evaluate_fused_gate(
        baseline_model, multi_head_model, evidential_model, eval_loader,
        attack_type=None, attack_params=None,
        **weights, threshold=thresh,
        max_boundary_samples=30
    )
    adv_acc_t, adv_def_t, adv_risk_t, _, _, _, _ = evaluate_fused_gate(
        baseline_model, multi_head_model, evidential_model, eval_loader,
        attack_type='pgd', attack_params=pgd_params,
        **weights, threshold=thresh,
        max_boundary_samples=30
    )
    print(f"{thresh:5.2f}     | {clean_acc_t:>7.2f}%  | {clean_def_t:>7.2f}%  | {clean_risk_t:>7.2f}%  | "
          f"{adv_acc_t:>7.2f}%  | {adv_def_t:>7.2f}%  | {adv_risk_t:>7.2f}%")

# ─── Save Fused Gate Results ───

import json

fused_results = {
    'weights': {k: float(v) for k, v in weights.items()},
    'threshold': float(fused_threshold),
    'clean_accuracy': float(clean_acc_fused),
    'clean_deferral_rate': float(clean_def_fused),
    'clean_risk': float(clean_risk_fused),
    'adversarial_accuracy': float(adv_acc_fused),
    'adversarial_deferral_rate': float(adv_def_fused),
    'adversarial_risk': float(adv_risk_fused),
    'num_deferred_clean': int(len(clean_deferred_fused)),
    'num_deferred_adversarial': int(len(adv_deferred_fused))
}

with open('fused_gate_results.json', 'w') as f:
    json.dump(fused_results, f, indent=2)
print("\n💾 Fused gate results saved to 'fused_gate_results.json'")

print("\n" + "="*55)
print("✅ Fused Gate Evaluation complete!")
print("="*55)


🔀 Fused Gate Evaluation

📊 Evaluating Fused Gate...


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [13]:
# ─────────────────────────────────────────────────────────────
# Part J: Final Results and Summary
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📋 COMPREHENSIVE EXPERIMENT SUMMARY")
print("="*60)

# ─── Overall Comparison Table ───

print("\n┌─────────────────────┬──────────────┬──────────────┬──────────────┬──────────────┐")
print("│ Model / Method      │   Clean Acc  │   Adv Acc    │   Deferral   │   Risk       │")
print("├─────────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤")

# Baseline
print(f"│ Baseline            │ {baseline_clean_acc:>11.2f}% │ {baseline_adv_acc:>11.2f}% │ {'':<12} │ {'':<12} │")
print(f"│ Baseline + Deferral │ {baseline_clean_def_acc:>11.2f}% │ {baseline_adv_def_acc:>11.2f}% │ {baseline_clean_def_rate:>11.2f}% │ {baseline_clean_risk:>11.2f}% │")

# Multi-Head
print(f"│ Multi-Head CNN      │ {multi_head_clean_acc:>11.2f}% │ {multi_head_adv_acc:>11.2f}% │ {'':<12} │ {'':<12} │")
print(f"│ Multi-Head + Def.   │ {multi_clean_def_acc:>11.2f}% │ {multi_adv_def_acc:>11.2f}% │ {multi_clean_def_rate:>11.2f}% │ {multi_clean_risk:>11.2f}% │")

# Evidential
print(f"│ Evidential CNN      │ {evidential_clean_acc:>11.2f}% │ {evidential_adv_acc:>11.2f}% │ {'':<12} │ {'':<12} │")
print(f"│ Evidential + Def.   │ {evid_clean_def_acc:>11.2f}% │ {evid_adv_def_acc:>11.2f}% │ {evid_clean_def_rate:>11.2f}% │ {evid_clean_risk:>11.2f}% │")

# Fused Gate
print(f"│ Fused Gate          │ {clean_acc_fused:>11.2f}% │ {adv_acc_fused:>11.2f}% │ {clean_def_fused:>11.2f}% │ {clean_risk_fused:>11.2f}% │")

print("└─────────────────────┴──────────────┴──────────────┴──────────────┴──────────────┘")

# ─── Key Findings ───

print("\n" + "="*55)
print("🔑 KEY FINDINGS")
print("="*55)

# Find best performing methods
best_clean_acc = max(baseline_clean_acc, multi_head_clean_acc, evidential_clean_acc, clean_acc_fused)
best_adv_acc = max(baseline_adv_acc, multi_head_adv_acc, evidential_adv_acc, adv_acc_fused)
best_clean_risk = min(baseline_clean_risk, multi_clean_risk, evid_clean_risk, clean_risk_fused)
best_adv_risk = min(baseline_adv_risk, multi_adv_risk, evid_adv_risk, adv_risk_fused)

best_clean_method = "Baseline" if best_clean_acc == baseline_clean_acc else \
                   "Multi-Head" if best_clean_acc == multi_head_clean_acc else \
                   "Evidential" if best_clean_acc == evidential_clean_acc else "Fused Gate"

best_adv_method = "Baseline" if best_adv_acc == baseline_adv_acc else \
                 "Multi-Head" if best_adv_acc == multi_head_adv_acc else \
                 "Evidential" if best_adv_acc == evidential_adv_acc else "Fused Gate"

best_clean_risk_method = "Baseline" if best_clean_risk == baseline_clean_risk else \
                        "Multi-Head" if best_clean_risk == multi_clean_risk else \
                        "Evidential" if best_clean_risk == evid_clean_risk else "Fused Gate"

best_adv_risk_method = "Baseline" if best_adv_risk == baseline_adv_risk else \
                      "Multi-Head" if best_adv_risk == multi_adv_risk else \
                      "Evidential" if best_adv_risk == evid_adv_risk else "Fused Gate"

print(f"\n📊 Best Clean Accuracy: {best_clean_acc:.2f}% ({best_clean_method})")
print(f"⚔️ Best Adversarial Accuracy: {best_adv_acc:.2f}% ({best_adv_method})")
print(f"🛡️ Best Clean Risk Reduction: {best_clean_risk:.2f}% ({best_clean_risk_method})")
print(f"🛡️ Best Adversarial Risk Reduction: {best_adv_risk:.2f}% ({best_adv_risk_method})")

# ─── Robustness Analysis ───

print("\n" + "="*55)
print("🛡️ ROBUSTNESS ANALYSIS")
print("="*55)

baseline_gap = baseline_clean_acc - baseline_adv_acc
multi_gap = multi_head_clean_acc - multi_head_adv_acc
evid_gap = evidential_clean_acc - evidential_adv_acc
fused_gap = clean_acc_fused - adv_acc_fused

print(f"\n📉 Clean vs Adversarial Accuracy Gap:")
print(f"  • Baseline: {baseline_gap:.2f}%")
print(f"  • Multi-Head CNN: {multi_gap:.2f}%")
print(f"  • Evidential CNN: {evid_gap:.2f}%")
print(f"  • Fused Gate: {fused_gap:.2f}%")

# ─── Deferral Effectiveness ───

print("\n" + "="*55)
print("📤 DEFERRAL EFFECTIVENESS")
print("="*55)

print(f"\nRisk Reduction with Deferral:")
print(f"  • Baseline Clean: {baseline_clean_risk:.2f}% (deferral: {baseline_clean_def_rate:.2f}%)")
print(f"  • Baseline Adv: {baseline_adv_risk:.2f}% (deferral: {baseline_adv_def_rate:.2f}%)")
print(f"  • Multi-Head Clean: {multi_clean_risk:.2f}% (deferral: {multi_clean_def_rate:.2f}%)")
print(f"  • Multi-Head Adv: {multi_adv_risk:.2f}% (deferral: {multi_adv_def_rate:.2f}%)")
print(f"  • Evidential Clean: {evid_clean_risk:.2f}% (deferral: {evid_clean_def_rate:.2f}%)")
print(f"  • Evidential Adv: {evid_adv_risk:.2f}% (deferral: {evid_adv_def_rate:.2f}%)")
print(f"  • Fused Gate Clean: {clean_risk_fused:.2f}% (deferral: {clean_def_fused:.2f}%)")
print(f"  • Fused Gate Adv: {adv_risk_fused:.2f}% (deferral: {adv_def_fused:.2f}%)")

# ─── Recommendations ───

print("\n" + "="*55)
print("💡 RECOMMENDATIONS")
print("="*55)

recommendations = []

if multi_head_clean_acc > evidential_clean_acc:
    recommendations.append("• Multi-Head CNN provides better clean accuracy than Evidential CNN")
if evidential_adv_acc > multi_head_adv_acc:
    recommendations.append("• Evidential CNN provides better adversarial robustness")
if clean_def_fused < multi_clean_def_rate and clean_acc_fused > multi_clean_def_acc:
    recommendations.append("• Fused Gate achieves better risk-accuracy tradeoff than individual methods")
if best_adv_risk_method == "Fused Gate":
    recommendations.append("• Fused Gate is recommended for adversarial scenarios")

print("\n".join(recommendations) if recommendations else "No specific recommendations available.")

# ─── Save Final Summary ───

final_summary = {
    'experiment_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'best_clean_accuracy': {
        'value': best_clean_acc,
        'method': best_clean_method
    },
    'best_adversarial_accuracy': {
        'value': best_adv_acc,
        'method': best_adv_method
    },
    'best_clean_risk': {
        'value': best_clean_risk,
        'method': best_clean_risk_method
    },
    'best_adversarial_risk': {
        'value': best_adv_risk,
        'method': best_adv_risk_method
    },
    'robustness_gaps': {
        'baseline': baseline_gap,
        'multi_head': multi_gap,
        'evidential': evid_gap,
        'fused_gate': fused_gap
    },
    'recommendations': recommendations
}

with open('final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)
print("\n💾 Final summary saved to 'final_summary.json'")

print("\n" + "="*55)
print("✅ Notebook Complete!")
print("="*55)



📋 COMPREHENSIVE EXPERIMENT SUMMARY

┌─────────────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ Model / Method      │   Clean Acc  │   Adv Acc    │   Deferral   │   Risk       │
├─────────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤


NameError: name 'baseline_adv_acc' is not defined